In [1]:
# Designing a daily, genre-level dataset that enables stakeholders to understand audience activity, content performance, and satisfaction.

In [2]:
# loading the silver interactions table
import pandas as pd

df_silver = pd.read_parquet("../data_processed/silver/silver_interactions.parquet")
pd.set_option('display.width', 300)
print(df_silver.head())

   userId  movieId  rating    rating_timestamp                   title     genres
0       1       31     2.5 2009-12-14 02:52:24  Dangerous Minds (1995)      Drama
1       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)  Animation
2       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)   Children
3       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)      Drama
4       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)    Musical


Expanding the dataset by synthetic data generation
Basic stats exploration in the silver layer revealed that the dataset is quite sparse and unlikely to yield really interesting analyses.
To address low event density in the source dataset, I'm introducing a synthetic data expansion step that duplicates and slightly perturbs interactions. This enables more realistic aggregation and visualization while preserving underlying behavioral patterns.

In [6]:
# Scale.
MULTIPLIER = 100  # 10–100 is good

# Create synthetic copies
import numpy as np

df_list = []

for i in range(MULTIPLIER):
    temp = df_silver.copy()
    
    # shift user IDs so they don’t collide
    temp["userId"] = temp["userId"] + (i * 100000)
    
    # add random time offset (within ~30 days and 2 hours) to rating timestamps to spread out interactions
    temp["rating_timestamp"] = temp["rating_timestamp"] + pd.to_timedelta(
        np.random.randint(-15, 15, size=len(temp)), unit="D") + pd.to_timedelta(
            np.random.randint(-60, 60, size=len(temp)), unit="m")
    
    
    # add rating noise for fun (and to prevent perfect duplicates)
    temp["rating"] = temp["rating"] + np.random.normal(0, 0.2, size=len(temp))
    temp["rating"] = temp["rating"].clip(0.5, 5.0)
    
    df_list.append(temp)

df_big = pd.concat(df_list, ignore_index=True)

print(df_big.head())





   userId  movieId    rating    rating_timestamp                   title     genres
0       1       31  2.597922 2009-12-17 03:28:24  Dangerous Minds (1995)      Drama
1       1     1029  2.883774 2009-12-13 02:23:59            Dumbo (1941)  Animation
2       1     1029  3.191872 2009-12-06 03:28:59            Dumbo (1941)   Children
3       1     1029  2.964010 2009-12-18 02:22:59            Dumbo (1941)      Drama
4       1     1029  3.071952 2009-12-10 02:12:59            Dumbo (1941)    Musical


Question we would like to answer with this dataset
What genres are trending over time?
Grain: month x genre


Metrics Definition
active_users: Number of unique users interacting with content in a given month
nunique(user_id)
engagement_events: Total number of rating events (using this as a proxy for user engagement)
count(rating)
avg_rating: Average rating given to content (using this as a proxy for user satisfaction)
mean(rating)
high_rating_pct: percentage of ratings equal to or above 4.0 (using this as an approximation of positive sentiment)
(rating >= 4).mean()
unique_titles_engaged: sum of unique movies
nunique(title)


In [7]:
# Add date and hour columns for easier aggregation in the gold table. The date column allows for analysis of daily trends, while the hour column will help us understand how engagement varies throughout the day.
df_big["date"] = df_big["rating_timestamp"].dt.floor("D")
df_big["hour"] = df_big["rating_timestamp"].dt.hour

# gold table implementation 

gold = df_big.groupby(["date", "hour", "genres"]).agg(
    active_users=("userId", "nunique"),
    engagement_events=("rating", "count"),
    avg_rating=("rating", "mean")
).reset_index()

print(gold.head())

        date  hour    genres  active_users  engagement_events  avg_rating
0 1994-12-25    10    Comedy             1                  1    3.276220
1 1994-12-25    10   Mystery             1                  1    5.000000
2 1994-12-25    11    Comedy             3                  3    3.043484
3 1994-12-25    11     Crime             4                  5    3.067323
4 1994-12-25    11  Thriller             4                  4    3.971012


In [8]:
gold.to_parquet("../data_processed/gold/audience_engagement_daily.parquet", index=False)